In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

In [10]:
working_directory = Path.cwd().parent.parent
print(working_directory)

CONSTRUCTOR_POINTS_PATH = working_directory / "data" / "clean" / "constructor_points.csv"
CONSTRUCTOR_PRICE_PATH = working_directory / "data" / "clean" / "constructor_price.csv"


/Users/jackguptill/Library/CloudStorage/OneDrive-Personal/Code/F1FantasyProject


In [11]:
PRED_YEAR = 2026
PRED_RACE_NUM = 1

RANDOM_STATE = 201

CLIP_NEGATIVE_PREDICTIONS = False

Ingest + Merge

In [12]:
def _standardize_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.str.strip()
                 .str.lower()
                 .str.replace(" ", "_")
                 .str.replace("-", "_")
    )
    return df

def _normalize_constructor_name(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
         .str.strip()
         .str.lower()
         .str.replace(r"\s+", " ", regex=True)
    )

def _coerce_year_race(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # allow either race or race_num
    if "race_num" not in df.columns and "race" in df.columns:
        df = df.rename(columns={"race": "race_num"})
    if "race" not in df.columns and "race_num" in df.columns:
        df["race"] = df["race_num"]

    if "year" in df.columns:
        df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    if "race_num" in df.columns:
        df["race_num"] = pd.to_numeric(df["race_num"], errors="coerce").astype("Int64")
    if "race" in df.columns:
        df["race"] = pd.to_numeric(df["race"], errors="coerce").astype("Int64")

    return df

def _find_constructor_col(df: pd.DataFrame) -> str:
    candidates = ["constructor", "constructor_name", "team", "team_name", "constructor_team"]
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"Couldn't find constructor/team column. Columns: {list(df.columns)}")

def load_constructor_points(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = _standardize_cols(df)
    df = _coerce_year_race(df)

    ctor_col = _find_constructor_col(df)
    if ctor_col != "constructor":
        df = df.rename(columns={ctor_col: "constructor"})
    df["constructor"] = _normalize_constructor_name(df["constructor"])

    if "constructor_points" not in df.columns:
        for alt in ["points", "total_points", "fantasy_points"]:
            if alt in df.columns:
                df = df.rename(columns={alt: "constructor_points"})
                break

    if "constructor_points" not in df.columns:
        raise KeyError(f"constructor_points not found. Columns: {list(df.columns)}")

    df["constructor_points"] = pd.to_numeric(df["constructor_points"], errors="coerce")

    keep = ["year", "race_num", "constructor", "constructor_points"]
    df = df[keep].drop_duplicates(subset=["year","race_num","constructor"]).reset_index(drop=True)
    return df

def load_constructor_price(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = _standardize_cols(df)
    df = _coerce_year_race(df)

    ctor_col = _find_constructor_col(df)
    if ctor_col != "constructor":
        df = df.rename(columns={ctor_col: "constructor"})
    df["constructor"] = _normalize_constructor_name(df["constructor"])

    if "price" not in df.columns:
        for alt in ["constructor_price", "cost", "value", "budget"]:
            if alt in df.columns:
                df = df.rename(columns={alt: "price"})
                break

    if "price" not in df.columns:
        raise KeyError(f"price not found. Columns: {list(df.columns)}")

    df["price"] = pd.to_numeric(df["price"], errors="coerce")

    keep = ["year", "race_num", "constructor", "price"]
    df = df[keep].drop_duplicates(subset=["year","race_num","constructor"]).reset_index(drop=True)
    return df

In [13]:
constructor_points = load_constructor_points(CONSTRUCTOR_POINTS_PATH)
constructor_price  = load_constructor_price(CONSTRUCTOR_PRICE_PATH)

constructor_base = constructor_price.merge(
    constructor_points,
    on=["year", "race_num", "constructor"],
    how="left",
    validate="1:1"
).sort_values(["year", "race_num", "constructor"]).reset_index(drop=True)

print("constructor_base created:", constructor_base.shape)
display(constructor_base.head())

constructor_base created: (711, 5)


,year,race_num,constructor,price,constructor_points
0,2023,1,alf,6.2,31.0
1,2023,1,alp,10.1,8.0
2,2023,1,alt,6.4,17.0
3,2023,1,ast,6.7,56.0
4,2023,1,fer,22.1,31.0


Feature Engineering

In [14]:
def add_constructor_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds rolling/history features per constructor:
      - lagged points, rolling means/stdev
      - season-to-date mean (safe transform approach)
      - price momentum features
    """
    out = df.copy()
    out = out.sort_values(["constructor", "year", "race_num"]).reset_index(drop=True)

    # ---- Points history features (use shift to avoid leakage) ----
    out["pts_lag1"] = out.groupby("constructor")["constructor_points"].shift(1)
    out["pts_lag2"] = out.groupby("constructor")["constructor_points"].shift(2)

    out["pts_roll3_mean"] = (
        out.groupby("constructor")["constructor_points"]
           .shift(1)
           .rolling(3, min_periods=1)
           .mean()
           .reset_index(level=0, drop=True)
    )
    out["pts_roll5_mean"] = (
        out.groupby("constructor")["constructor_points"]
           .shift(1)
           .rolling(5, min_periods=1)
           .mean()
           .reset_index(level=0, drop=True)
    )
    out["pts_roll5_std"] = (
        out.groupby("constructor")["constructor_points"]
           .shift(1)
           .rolling(5, min_periods=2)
           .std()
           .reset_index(level=0, drop=True)
    )

    # ---- Season-to-date mean (excluding current race) ----
    # Compute shifted cumulative mean per (constructor, year)
    g = out.groupby(["constructor", "year"])["constructor_points"]
    out["season_pts_to_date_mean"] = g.transform(lambda s: s.shift(1).expanding(min_periods=1).mean())

    # ---- Price features ----
    out["price_lag1"] = out.groupby("constructor")["price"].shift(1)
    out["price_change_1"] = out["price"] - out["price_lag1"]

    out["price_roll3_mean"] = (
        out.groupby("constructor")["price"]
           .shift(1)
           .rolling(3, min_periods=1)
           .mean()
           .reset_index(level=0, drop=True)
    )

    # ---- Simple time features ----
    out["race_num_int"] = out["race_num"].astype("Int64").astype(float)
    out["year_int"] = out["year"].astype("Int64").astype(float)
    out["time_index"] = out["year_int"] * 100 + out["race_num_int"]

    return out

In [15]:
df_feat = add_constructor_features(constructor_base)

Train / Predict Splits

In [16]:
# "Prediction rows" are price rows where constructor_points is NA (should include 2026 race 1)
pred_mask = (df_feat["year"] == PRED_YEAR) & (df_feat["race_num"] == PRED_RACE_NUM)

# Training data: anything with a known target AND not the prediction rows
train_df = df_feat[~pred_mask].copy()
train_df = train_df[train_df["constructor_points"].notna()].copy()

pred_df = df_feat[pred_mask].copy()

print("train_df shape:", train_df.shape)
print("pred_df  shape:", pred_df.shape)
display(train_df.head(3))
display(pred_df.head(10))

train_df shape: (700, 17)
pred_df  shape: (11, 17)


,year,race_num,constructor,price,constructor_points,pts_lag1,pts_lag2,pts_roll3_mean,pts_roll5_mean,pts_roll5_std,season_pts_to_date_mean,price_lag1,price_change_1,price_roll3_mean,race_num_int,year_int,time_index
0,2023,1,alf,6.2,31.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2023.0,202301.0
1,2023,2,alf,6.2,7.0,31.0,NaN,31.0,31.0,NaN,31.0,6.2,0.0,6.2,2.0,2023.0,202302.0
2,2023,3,alf,6.1,26.0,7.0,31.0,19.0,19.0,16.970563,19.0,6.2,-0.1,6.2,3.0,2023.0,202303.0


,year,race_num,constructor,price,constructor_points,pts_lag1,pts_lag2,pts_roll3_mean,pts_roll5_mean,pts_roll5_std,season_pts_to_date_mean,price_lag1,price_change_1,price_roll3_mean,race_num_int,year_int,time_index
92,2026,1,alp,12.5,NaN,4.0,14.0,7.000000,9.400000,5.458938,NaN,10.1,2.4,9.900000,1.0,2026.0,202601.0
115,2026,1,amr,10.3,NaN,NaN,NaN,28.500000,33.250000,6.701990,NaN,NaN,NaN,7.950000,1.0,2026.0,202601.0
186,2026,1,aud,6.6,NaN,NaN,NaN,6.500000,6.000000,12.192894,NaN,NaN,NaN,10.800000,1.0,2026.0,202601.0
187,2026,1,cad,6.0,NaN,NaN,NaN,21.000000,7.333333,14.571662,NaN,NaN,NaN,10.500000,1.0,2026.0,202601.0
258,2026,1,fer,23.3,NaN,83.0,22.0,51.000000,41.600000,33.155693,NaN,31.8,-8.5,31.833333,1.0,2026.0,202601.0
329,2026,1,has,7.4,NaN,NaN,NaN,9.500000,29.500000,26.210685,NaN,NaN,NaN,14.700000,1.0,2026.0,202601.0
448,2026,1,mcl,28.9,NaN,77.0,96.0,46.333333,57.600000,52.624139,NaN,36.1,-7.2,36.000000,1.0,2026.0,202601.0
519,2026,1,mer,29.3,NaN,35.0,57.0,56.333333,63.600000,21.349473,NaN,28.4,0.9,28.100000,1.0,2026.0,202601.0
520,2026,1,rb,6.3,NaN,NaN,NaN,46.000000,65.000000,24.385788,NaN,NaN,NaN,28.250000,1.0,2026.0,202601.0
521,2026,1,rbr,28.2,NaN,NaN,NaN,35.000000,56.333333,21.007935,NaN,NaN,NaN,28.400000,1.0,2026.0,202601.0


Define Features + Pipeline

In [17]:
TARGET = "constructor_points"

# numeric features (keep it explicit & simple)
numeric_features = [
    "price",
    "price_lag1",
    "price_change_1",
    "price_roll3_mean",
    "pts_lag1",
    "pts_lag2",
    "pts_roll3_mean",
    "pts_roll5_mean",
    "pts_roll5_std",
    "season_pts_to_date_mean",
    "race_num_int",
    "year_int",
]

categorical_features = [
    "constructor",
]

# Filter down to existing columns (so the notebook doesn't die if one doesn't exist)
numeric_features = [c for c in numeric_features if c in train_df.columns]
categorical_features = [c for c in categorical_features if c in train_df.columns]

X = train_df[numeric_features + categorical_features].copy()
y = train_df[TARGET].astype(float).copy()

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]), numeric_features),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_features),
    ],
    remainder="drop"
)

# Candidate models (fast + decent)
models = {
    "ridge": Ridge(random_state=RANDOM_STATE) if "random_state" in Ridge().get_params() else Ridge(),
    "elasticnet": ElasticNet(random_state=RANDOM_STATE, max_iter=10000),
    "rf": RandomForestRegressor(
        n_estimators=600,
        random_state=RANDOM_STATE,
        min_samples_leaf=2,
        n_jobs=-1
    ),
    "hgb": HistGradientBoostingRegressor(
        random_state=RANDOM_STATE,
        max_depth=6,
        learning_rate=0.06,
        max_iter=600
    )
}


Time-series CV to choose best model

In [18]:
# Sort by time_index so TimeSeriesSplit behaves correctly
train_df_sorted = train_df.sort_values(["year", "race_num", "constructor"]).reset_index(drop=True)
X_sorted = train_df_sorted[numeric_features + categorical_features].copy()
y_sorted = train_df_sorted[TARGET].astype(float).copy()

tscv = TimeSeriesSplit(n_splits=5)

scores = []
for name, model in models.items():
    pipe = Pipeline(steps=[("preprocess", preprocess), ("model", model)])
    # Use MAE (negated by sklearn)
    cv_mae = -cross_val_score(pipe, X_sorted, y_sorted, cv=tscv, scoring="neg_mean_absolute_error", n_jobs=None).mean()
    scores.append((name, cv_mae))

scores = sorted(scores, key=lambda x: x[1])
print("CV MAE (lower is better):")
for name, mae in scores:
    print(f"  {name:10s}  MAE={mae:.3f}")

best_name = scores[0][0]
best_model = models[best_name]
best_pipe = Pipeline(steps=[("preprocess", preprocess), ("model", best_model)])

print("\nBest model:", best_name)

CV MAE (lower is better):
  elasticnet  MAE=15.858
  rf          MAE=17.409
  ridge       MAE=17.855
  hgb         MAE=19.522

Best model: elasticnet


Fit on full training set + quick holdout check

In [19]:
# Quick holdout: last ~20% by time_index
train_df_sorted["time_index"] = train_df_sorted["year"].astype(int) * 100 + train_df_sorted["race_num"].astype(int)
cutoff = np.quantile(train_df_sorted["time_index"], 0.80)

train_part = train_df_sorted[train_df_sorted["time_index"] <= cutoff].copy()
valid_part = train_df_sorted[train_df_sorted["time_index"] > cutoff].copy()

X_tr = train_part[numeric_features + categorical_features]
y_tr = train_part[TARGET].astype(float)

X_va = valid_part[numeric_features + categorical_features]
y_va = valid_part[TARGET].astype(float)

best_pipe.fit(X_tr, y_tr)
va_pred = best_pipe.predict(X_va)

va_mae = mean_absolute_error(y_va, va_pred)
va_rmse = mean_squared_error(y_va, va_pred) ** 0.5

print(f"\nHoldout (last ~20%) MAE:  {va_mae:.3f}")
print(f"Holdout (last ~20%) RMSE: {va_rmse:.3f}")

# Fit final on ALL training
best_pipe.fit(X, y)


Holdout (last ~20%) MAE:  17.801
Holdout (last ~20%) RMSE: 24.256


,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


Predict current week

In [20]:
X_pred = pred_df[numeric_features + categorical_features].copy()
pred_points = best_pipe.predict(X_pred)

if CLIP_NEGATIVE_PREDICTIONS:
    pred_points = np.maximum(pred_points, 0)

current_week_constructor_preds = pred_df[["year", "race_num", "constructor", "price"]].copy()
current_week_constructor_preds["predicted_points"] = pred_points

current_week_constructor_preds = current_week_constructor_preds.sort_values(
    "predicted_points", ascending=False
).reset_index(drop=True)

display(current_week_constructor_preds)

,year,race_num,constructor,price,predicted_points
0,2026,1,mcl,28.9,53.242756
1,2026,1,mer,29.3,51.584534
2,2026,1,rbr,28.2,48.359364
3,2026,1,fer,23.3,40.582527
4,2026,1,rb,6.3,26.878194
5,2026,1,amr,10.3,22.922543
6,2026,1,wil,12.0,22.280189
7,2026,1,has,7.4,19.857089
8,2026,1,alp,12.5,16.529915
9,2026,1,aud,6.6,10.462140


Save Outputs

## Export predicted points


#### read in the driver predictions file


In [25]:
in_path = working_directory / "data" / "predictions" / "constructors" / f"constructor_preds_2026.csv"
hist_predictions = pd.read_csv(in_path)

#### append the current preds to the histrical predictions

In [26]:
new_hist_preds = pd.concat([current_week_constructor_preds, hist_predictions],axis=0, ignore_index = True)

#### export the new file

In [27]:
out_path = working_directory / "data" / "predictions" / "constructors" / f"constructor_preds_2026.csv"

new_hist_preds.to_csv(out_path)
